# v46 — a head for the GPT-5.6 trio

v44 scored **0.626**. v45 added stylegram and scored 0.624, so stylegram is out: it is a four-view
TF-IDF readout and our SVC is already a four-view TF-IDF readout, so it duplicated a member instead
of adding a view.

The out-of-fold confusion said where the points actually are. `gpt-5.6-sol` loses 20% of its
passages to `terra` and 20% to `luna`; `terra` loses 15% to `luna` and 13% to `sol`. All three sit at
0.31 recall and all the confusion is *inside one family*, which is why another general member cannot
fix them.

This version adds one thing: a head that only ever sees those three, trained on features centred
inside each prompt group. The pool already has several models answering the same prompt, so the
pairs exist without generating anything. It is measured on a prompt-grouped holdout first and only
applied if it clearly beats the SVC on the same rows.

In [ ]:
import gc, glob, json, os, re, shutil, time

# The scoring run has no internet, so transformers must never try to reach the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn

# Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights, from the public part-two notebook's out-of-fold fit.
W_SVC = 1.0
LORAS = {
    "lora_qwen3_14b":  dict(w=0.4, twin_w=0.1, bs=8, base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
    "lora_qwen25_14b": dict(w=0.2, twin_w=0.1, bs=8, base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
}
W_DEBERTA = 0.25
TWIN = dict(thresh=0.85, min_size=2, max_size=18)
# Adaptive SVC topic correction (the 0.639 public stack's mechanism): inside a topic cluster,
# subtract the cluster's leave-one-out mean from the SVC log probabilities, gated by how unsure
# the passage is. lam=0 disables it and reproduces v42 exactly.
CORRECTION = dict(thresh=0.95, min_size=8, max_size=18, lam=0.62)

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)
LABEL2I = {l: i for i, l in enumerate(LABELS)}


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    """Add one member's weighted log probabilities to the running score."""
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map(LABEL2I).values
LOGIT = np.zeros((len(test), K))
REPORT = {}
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

## 1. Clusters, the n-gram SVC, the adaptive correction, and the OOF confusion

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import hstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\\w+|[^\\w\\s]", t))


pool = pd.read_parquet(find("pool_v4.parquet"))
pool = pool.dropna(subset=["text", "model"]).copy()
pool["text"] = pool["text"].astype(str)
pool["model"] = pool["model"].astype(str)
pool = pool[pool["model"].isin(LABELS)].copy()
train_texts = train.text.astype(str).tolist()
test_texts = test.text.astype(str).tolist()
print(f"pool_v4: {len(pool):,} passages", flush=True)

# ---------------------------------------------------------------------------
# Topic clusters, computed once. Two thresholds: the tight one drives the centred
# twin, the looser one feeds the adaptive SVC topic correction (which needs
# clusters of 8 to 18 passages to have a meaningful mean to subtract).
# ---------------------------------------------------------------------------
if len(test_texts) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test_texts)
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    Z = linkage(squareform(D, checks=False), method="average")
    TEST_GROUPS_TWIN = fcluster(Z, TWIN["thresh"], criterion="distance")
    TEST_GROUPS_CORR = fcluster(Z, CORRECTION["thresh"], criterion="distance")
    del D, Xt, Z
else:
    TEST_GROUPS_TWIN = np.ones(len(test_texts), dtype=int)
    TEST_GROUPS_CORR = np.ones(len(test_texts), dtype=int)
size = np.bincount(TEST_GROUPS_TWIN)[TEST_GROUPS_TWIN]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
corr_size = np.bincount(TEST_GROUPS_CORR)[TEST_GROUPS_CORR]
print(f"twin clusters: {int((size == 1).sum())} passages alone, {int(twin_rows.sum())} in 2-18", flush=True)
print(f"correction clusters: {int((corr_size == 1).sum())} alone, "
      f"{int(((corr_size >= CORRECTION['min_size']) & (corr_size <= CORRECTION['max_size'])).sum())} "
      f"in {CORRECTION['min_size']}-{CORRECTION['max_size']}, "
      f"max size {int(corr_size.max())}", flush=True)

fit_texts = train_texts + pool.text.tolist()
fit_y = np.r_[y, pool.model.map(LABEL2I).values].astype(int)
_g = next((c for c in ("prompt_id", "prompt_key", "prompt") if c in pool.columns), None)
fit_groups = np.r_[
    (train["prompt_id"].astype(str).values if "prompt_id" in train else np.arange(N_TR).astype(str)),
    (pool[_g].astype(str).values if _g else np.array([f"pool{i}" for i in range(len(pool))]))]
print(f"pool prompt key: {_g or 'none — pool rows are singletons'}", flush=True)
N_TR = len(train_texts)

A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)
    analyzer = "char" if view == "shape" else "word" if view == "small" else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
    B.append(vec.transform([prep(t) for t in test_texts]) * weight)
X_CHAR = A[0].tocsr()            # the character view alone, reused by the trio head
X_CHAR_TEST = B[0].tocsr()
X_FIT = hstack(A, format="csr")
X_TEST = hstack(B, format="csr")
del A, B
gc.collect()

svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(X_FIT, fit_y)
LOG_SVC = np.log(np.clip(softmax(svc.decision_function(X_TEST)), 1e-9, 1))
print(f"SVC fitted on {len(fit_texts):,} rows ({time.time() - T0:.0f}s)", flush=True)
del svc
gc.collect()


def adaptive_topic_correction(logp, groups_arr, lam, min_size, max_size):
    """Subtract each topic cluster's leave-one-out mean from the SVC's log probabilities.

    A prompt about physics pushes some classes up for every passage that answers it. That is
    topic, not author. Inside a cluster of min_size..max_size passages we remove it, but only
    as much as the passage is unsure: gate = clip((2 - (top1 - top2)) / 1.6, 0.15, 1).
    """
    out = logp.copy()
    touched = 0
    for g in np.unique(groups_arr):
        idx = np.flatnonzero(groups_arr == g)
        n = len(idx)
        if not (min_size <= n <= max_size):
            continue
        block = logp[idx]
        loo = (block.sum(0) - block) / (n - 1)
        top1 = block.max(1)
        top2 = np.sort(block, 1)[:, -2]
        gate = np.clip((2.0 - (top1 - top2)) / 1.6, 0.15, 1.0)
        out[idx] = block - lam * gate[:, None] * loo
        touched += n
    return out, touched


CORRECTED_ROWS = 0
if len(test_texts) >= 2 and CORRECTION["lam"] > 0:
    LOG_SVC, CORRECTED_ROWS = adaptive_topic_correction(
        LOG_SVC, TEST_GROUPS_CORR, CORRECTION["lam"], CORRECTION["min_size"], CORRECTION["max_size"])
print(f"adaptive topic correction touched {CORRECTED_ROWS} passages", flush=True)

vote("svc", softmax(LOG_SVC), W_SVC)

# ---------------------------------------------------------------------------
# Diagnostic, reporting only: who steals from the weakest classes.
# Two prompt-grouped folds of the production SVC (fold train rows + the whole pool),
# then the per-class recall and, for each weak class, where its passages actually go.
# The blend cannot be scored out of fold here (the LoRA stage two saw the train rows),
# so this is the strongest single text-only member's confusion, which is what we can
# measure for free and what the next modelling decision needs.
# ---------------------------------------------------------------------------
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold

if len(train) >= 500:
    oof_groups = train["prompt_id"].astype(str).values if "prompt_id" in train else np.arange(N_TR).astype(str)
    pool_rows = np.arange(N_TR, len(fit_texts))
    oof = np.zeros((N_TR, K), np.float32)
    for fi, (itr, iva) in enumerate(GroupKFold(2).split(np.arange(N_TR), y, oof_groups), 1):
        rows = np.r_[itr, pool_rows]
        m = LinearSVC(C=0.5, random_state=100 + fi, max_iter=5000, class_weight="balanced")
        m.fit(X_FIT[rows], fit_y[rows])
        oof[iva] = softmax(m.decision_function(X_FIT[iva]))
        print(f"OOF SVC fold {fi}/2 done ({time.time() - T0:.0f}s)", flush=True)
        del m
        gc.collect()

    pred = oof.argmax(1)
    rec = {LABELS[i]: float((pred[y == i] == i).mean()) for i in range(K)}
    REPORT["oof_svc_ba_2fold"] = float(balanced_accuracy_score(y, pred))
    REPORT["oof_per_class_recall"] = {k: round(v, 4) for k, v in rec.items()}
    weak = sorted(rec, key=rec.get)[:8]
    conf = {}
    for lab in weak:
        i = LABEL2I[lab]
        rows = y == i
        cnt = np.bincount(pred[rows], minlength=K)
        top = np.argsort(cnt)[::-1][:4]
        conf[lab] = {"recall": round(rec[lab], 3),
                     "goes_to": {LABELS[j]: round(float(cnt[j] / max(rows.sum(), 1)), 3) for j in top}}
    REPORT["weakest_confusion"] = conf
    print(f"\nOOF SVC BA (2 folds) = {REPORT['oof_svc_ba_2fold']:.4f}", flush=True)
    print("weakest classes and where their passages go:", flush=True)
    for lab in weak:
        line = ", ".join(f"{k} {v:.2f}" for k, v in conf[lab]["goes_to"].items())
        print(f"  {lab:<24} recall {conf[lab]['recall']:.3f} -> {line}", flush=True)
else:
    print("interactive run: skipping the OOF confusion diagnostic", flush=True)

del X_FIT
gc.collect()

## 2. LoRA members

In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    """The attached base model whose config.json matches spec."""
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """The first keep decoder layers, split over both T4s when they do not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    H, I, V = mc.hidden_size, mc.intermediate_size, mc.vocab_size
    hd = getattr(mc, "head_dim", H // mc.num_attention_heads)
    q, kv = mc.num_attention_heads * hd, mc.num_key_value_heads * hd
    gib = 2 * (V * H + keep * (2 * H * q + 2 * H * kv + 3 * H * I)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()   # the checkpoints were trained without the final norm
    print(f"trunk of {keep} layers, {gib:.1f} GiB of weights", flush=True)
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    """W becomes W + B A alpha / r for every adapted linear layer."""
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." not in k:
            continue
        name = k.split(".lora_A.")[0]
        lin = modules[name[name.index("layers."):]]
        Bm = sd[k.replace(".lora_A.", ".lora_B.")]
        delta = (Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale
        lin.weight.add_(delta.to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z, one row per passage, and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    # The end token from training, not this tokenizer's. Kaggle's Qwen3-14B-Base carries the
    # chat model's end token (151645 instead of 151643), which silently changes ~14% of its
    # top predictions. The checkpoint's meta.json records the right id.
    eos_id = cfg.get("eos_id", tok.eos_token_id)
    print(f"end token {eos_id}, this tokenizer's is {tok.eos_token_id}", flush=True)
    ids = [x + [eos_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together means less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)

    def embed(idx):
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow, so redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token soaks up attention whatever it is, so leave it out of the average
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        try:
            embed(idx)
        except torch.cuda.OutOfMemoryError:   # the longest batch can overflow a 15 GB T4
            torch.cuda.empty_cache()
            for i in idx:
                embed([i])
        if s % (bs * 25) == 0:
            print(f"  {s + len(idx)}/{len(ids)} passages ({time.time() - T0:.0f}s)", flush=True)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}

## 3. LoRA inference and the centred twin

In [ ]:
for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z, heads = lora_features(ckpt, find_base(mc["base"]), test_texts, mc["bs"])
    vote(ckpt_name, softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), mc["w"])
    if twin_rows.any():
        Zc = Z.copy()
        for g in np.unique(TEST_GROUPS_TWIN[twin_rows]):
            Zc[TEST_GROUPS_TWIN == g] -= Zc[TEST_GROUPS_TWIN == g].mean(0)
        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
        vote(ckpt_name + " twin", softmax(centred), mc["twin_w"], rows=twin_rows)
    del Z
    gc.collect()

## 4. DeBERTa

In [ ]:
ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):   # one long copy to local disk beats many small reads from the dataset mount
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(test_texts, truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
with torch.inference_mode():
    for s in range(0, len(order), 32):
        idx = order[s:s + 32]
        bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
del enc, Z
gc.collect()
torch.cuda.empty_cache()

## 5. The trio head (new)

In [ ]:
"""GPT-5.6 trio head: the three siblings that keep eating each other.

The out-of-fold confusion on the SVC says it plainly: gpt-5.6-sol loses 20% to terra and 20% to
luna, terra loses 15% to luna and 13% to sol. Three classes at 0.31 recall, all confusion *inside*
one family, which is why adding another general member changes nothing.

So this trains a head that only ever sees those three, on features centred inside each prompt group
(the pool already stores several models answering the same prompt, so the pairs exist without
generating anything). Style is what is left after the prompt's own topic is subtracted.

It is measured before it is trusted: a prompt-grouped split reports the head's accuracy against the
SVC's on exactly the same rows. If the head is not clearly better, it is not applied.
"""
from sklearn.linear_model import LogisticRegression

TRIO = ["gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra"]
TRIO_IDX = np.array([LABEL2I[t] for t in TRIO])
TRIO_W = 0.35                 # how much of the head's log-probability to add
TRIO_REPORT = {"status": "not run"}


def centre_within(mat, groups):
    """Subtract each prompt group's mean, so only the style difference survives."""
    out = mat.astype(np.float32).tolil()
    out = mat.astype(np.float32).copy()
    touched = 0
    for g in np.unique(groups):
        idx = np.flatnonzero(groups == g)
        if len(idx) < 2:
            continue
        m = np.asarray(mat[idx].mean(0), dtype=np.float32)
        out[idx] = mat[idx] - m
        touched += len(idx)
    return out.tocsr(), touched


trio_rows = np.flatnonzero(np.isin(fit_y, TRIO_IDX))
trio_groups = fit_groups[trio_rows]
sizes = np.array([(trio_groups == g).sum() for g in trio_groups])
print(f"trio rows: {len(trio_rows):,} | rows inside a prompt group of 2+: {(sizes >= 2).sum():,}", flush=True)

if len(trio_rows) < 200 or (sizes >= 2).sum() < 100:
    TRIO_REPORT["status"] = "skipped: too few paired trio rows in the pool"
    print("trio head SKIPPED:", TRIO_REPORT["status"], flush=True)
else:
    # prompt-grouped split, so the head is measured on prompts it has never seen
    from sklearn.model_selection import GroupShuffleSplit
    tr_i, va_i = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
                      .split(trio_rows, groups=trio_groups))
    Xc, _ = centre_within(X_CHAR[trio_rows], trio_groups)
    yt = fit_y[trio_rows]
    head = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced", n_jobs=-1)
    head.fit(Xc[tr_i], yt[tr_i])
    head_va = head.predict(Xc[va_i])
    head_acc = float((head_va == yt[va_i]).mean())
    svc_va = oof[trio_rows[va_i]].argmax(1) if "oof" in globals() else None
    svc_acc = float((svc_va == yt[va_i]).mean()) if svc_va is not None else None
    print(f"trio head: holdout accuracy {head_acc:.4f}"
          + (f" vs SVC {svc_acc:.4f} on the same rows" if svc_acc is not None else ""), flush=True)
    TRIO_REPORT.update({"status": "trained", "holdout_acc": round(head_acc, 4),
                        "svc_acc_same_rows": None if svc_acc is None else round(svc_acc, 4),
                        "rows": int(len(trio_rows)), "weight": TRIO_W})

    # apply on the test side: centre inside each test cluster, then only touch rows whose
    # blended top-1 is already one of the three
    if svc_acc is not None and head_acc > svc_acc + 0.02:
        Xt_c, _ = centre_within(X_CHAR_TEST, TEST_GROUPS_TWIN)
        P_trio = head.predict_proba(Xt_c)
        top1 = LOGIT.argmax(1)
        hit = np.isin(top1, TRIO_IDX)
        for j, cls in enumerate(head.classes_):
            LOGIT[hit, cls] += TRIO_W * np.log(np.clip(P_trio[hit, j], 1e-9, 1.0))
        TRIO_REPORT["rows_touched"] = int(hit.sum())
        print(f"trio head applied to {int(hit.sum())} test rows (weight {TRIO_W})", flush=True)
    else:
        TRIO_REPORT["status"] = "trained but not applied (no clear gain over the SVC)"
        print("trio head not applied — the head does not beat the SVC by 0.02", flush=True)

## 6. Blend, Sinkhorn, submit

In [ ]:
def sinkhorn(P, iters=300):
    """One scale per class, so that every class gets an equal share of the probability."""
    c = np.ones(K)
    target = np.full(K, len(P) / K)
    for _ in range(iters):
        R = P * c
        R /= R.sum(1, keepdims=True)
        c *= target / R.sum(0)
    R = P * c
    return R / R.sum(1, keepdims=True)


P = softmax(LOGIT)
if len(test) >= 50:   # skip on the 3 row public test.csv
    P = sinkhorn(P)

sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all() and sub["id"].is_unique
sub.to_parquet("submission.parquet", index=False)

REPORT.update({
    "recipe": "v42 part-two reproduction + adaptive SVC topic correction",
    "correction": CORRECTION,
    "correction_rows": int(CORRECTED_ROWS),
    "weights": {"svc": W_SVC, **{k: v["w"] for k, v in LORAS.items()}, "deberta": W_DEBERTA},
    "rows": {"train": int(N_TR), "pool_v4": int(len(pool)), "test": int(len(test_texts))},
    "sinkhorn_iters": 300 if len(test) >= 50 else 0,
    "twin_rows": int(twin_rows.sum()),
})
with open("validation_results.json", "w") as f:
    json.dump(REPORT, f, indent=2)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")